# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [162]:
# TODO: // DONE Fill this in before submitting.
GROUP_ID = 11
NOTEBOOK_URL = "https://github.com/arion023/tbd-workshop-1/blob/master/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    "Marcin Kowalczyk / 318677",
    "Alicja Jurzysta / 318614",
    "Kacper Pawłowski / 321141",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [163]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [164]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.12.3
Polars: 1.41.2
Pandas: 3.0.3
DuckDB: 1.5.3
CPU logical cores: 16
RAM GiB: 31.24


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [165]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'Delivery/logistics events',
 'feature': 'late status updates',
 'stress': 'late events and time windows'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [243]:
# TODO: //DONE
# Choose the main dataset scale for your final benchmark and verify output paths before generation.
# N_ROWS is the main row count reported for this notebook. Extra row counts are optional stress tests.
# Dataset configuration
SCALE = "medium"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
    "extra_large" : 250_000_000
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

# Leave SEED as None if you want independent data on each generation.
# If you need to reproduce exactly the same dataset later, set SEED to the value stored in the manifest.
SEED = None
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 11 {'name': 'Delivery/logistics events', 'feature': 'late status updates', 'stress': 'late events and time windows'}
Rows: 10000000
Run seed recorded in manifest: 92602154070936546290147030962673889752
Output directory: ../data/phase2_26L/group_11


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [167]:
# TODO: // DONE 
# Adapt customize_for_variant(...) and generate_dimension_table(...) to your variant.
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


def customize_for_variant(df, card, rng):
    # TODO: // DONE 
    # Adapt this function to your assigned variant.
    # Examples of acceptable changes:
    # - rename entity_id to user_id, device_id, order_id, ticket_id, etc.
    # - add domain-specific categorical columns,
    # - add one or two numeric columns that make sense for your domain,
    # - introduce skew, nulls, rare categories, or late events,
    # - add a small dimension table for a join query.
    
    n = df.height

    statuses = rng.choice(
        ["picked_up", "in_transit", "out_for_delivery", "delivered", "delayed"], 
        size=n, 
        p=[0.10, 0.40, 0.15, 0.30, 0.05]
    )

    delays = np.zeros(n, dtype=int)
    delayed_mask = (statuses == "delayed")
    delays[delayed_mask] = rng.integers(15, 1440, size=delayed_mask.sum())
    warehouse_ids = rng.integers(1, 1001, size=n)

    return (
        df
        .rename({
            "entity_id": "tracking_number",
            "metric_1": "package_weight_kg",
            "metric_2": "delivery_cost"
        })
        .with_columns([
            pl.Series("status", statuses),
            pl.Series("delay_minutes", delays),
            pl.Series("warehouse_id", warehouse_ids)
        ])
        .drop("category")
    )


def generate_dimension_table(card, rng):
    # TODO: // DONE
    # Replace this generic dimension table with something meaningful for your variant.
    # It can describe products, campaigns, devices, courses, tickets, routes, alerts, etc.
    n_warehouses = 1_000
    
    regions = rng.choice(
        ["North", "South", "East", "West", "Central"], 
        size=n_warehouses
    )

    capacity = rng.integers(5_000, 50_000, size=n_warehouses)

    return pl.DataFrame(
        {
            "warehouse_id": np.arange(1, n_warehouses + 1),
            "warehouse_region": regions,
            "warehouse_capacity": capacity,
        })

In [168]:
# TODO: //DONE
# Run this after adapting the generator. Verify that generated data is not committed to Git.
# Generate and save the dataset
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)
dimension = generate_dimension_table(CARD, rng)

events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Optional partitioned layout for experiments with predicate pushdown and file layout.
events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

In [169]:
# TODO: // DONE 
# Create an optimized Parquet layout for one selected query pattern.
# Example ideas:
# - sort by columns used in range filters before writing,
# - choose a smaller row_group_size if it improves row-group pruning,
# - partition by date or another selective filter column,
# - add bloom filters only if your chosen writer and reader expose this option clearly.
# Replace the sort columns with columns from your own query pattern.
events.sort(["event_date", "status"]).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000,
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))


{
  "created_at_utc": "2026-06-14T15:35:33.679847+00:00",
  "group_id": 11,
  "variant": {
    "name": "Delivery/logistics events",
    "feature": "late status updates",
    "stress": "late events and time windows"
  },
  "scale": "medium",
  "rows": 10000000,
  "run_seed": 267121021755925499556232552786502653037,
  "paths": {
    "events": "../data/phase2_26L/group_11/events.parquet",
    "events_partitioned": "../data/phase2_26L/group_11/events_partitioned",
    "events_optimized": "../data/phase2_26L/group_11/events_optimized.parquet",
    "dimension": "../data/phase2_26L/group_11/dimension.parquet"
  },
  "environment": {
    "python": "3.12.3",
    "polars": "1.41.2",
    "pandas": "3.0.3",
    "duckdb": "1.5.3",
    "cpu_logical_cores": 16,
    "ram_gib": 31.24
  }
}


## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [170]:
# TODO: //DONE
# Inspect schema, row count, null counts, and basic category distributions.
# Keep this section short, but include enough evidence that your data was generated correctly.

df_events = pl.read_parquet(EVENTS_PATH)
df_dim = pl.read_parquet(DIMENSION_PATH)

print("Events schema:")
print(df_events.schema)

print("Dimension schema:")
print(df_dim.schema)

print(f"Event number: {df_events.height:,}")
print(f"Warehouse number: {df_dim.height:,}")
print("-" * 50)

print("Dimension values:")
print(df_dim["warehouse_region"].value_counts())

print("Status distribution:")
print(df_events["status"].value_counts())


print("Delayed time:")
print(df_events["delay_minutes"].hist())

delayed_stats = (
    df_events
    .filter(pl.col("status") == "delayed")
    .select(
        pl.col("delay_minutes").min().alias("min_delay"),
        pl.col("delay_minutes").max().alias("max_delay"),
        pl.col("delay_minutes").mean().alias("avg_delay")
    )
)
display(delayed_stats)

Events schema:
Schema({'event_id': Int64, 'tracking_number': Int64, 'event_ts': Datetime(time_unit='us', time_zone=None), 'country': String, 'device': String, 'package_weight_kg': Float64, 'delivery_cost': Int64, 'tags': List(String), 'event_date': Date, 'status': String, 'delay_minutes': Int64, 'warehouse_id': Int64})
Dimension schema:
Schema({'warehouse_id': Int64, 'warehouse_region': String, 'warehouse_capacity': Int64})
Event number: 10,000,000
Warehouse number: 1,000
--------------------------------------------------
Dimension values:
shape: (5, 2)
┌──────────────────┬───────┐
│ warehouse_region ┆ count │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ South            ┆ 206   │
│ East             ┆ 199   │
│ Central          ┆ 220   │
│ West             ┆ 175   │
│ North            ┆ 200   │
└──────────────────┴───────┘
Status distribution:
shape: (5, 2)
┌──────────────────┬─────────┐
│ status           ┆ count   │
│ ---              ┆ ---

min_delay,max_delay,avg_delay
i64,i64,f64
15,1439,726.802948


In [171]:
print("Columns:")
print(df_events.columns)

print("\nStatus distribution:")
display(
    df_events
    .group_by("status")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / df_events.height * 100).round(2).alias("percent"))
    .sort("count", descending=True)
)

print("\nNull counts:")
display(
    df_events
    .null_count()
    .transpose(include_header=True, header_name="column", column_names=["null_count"])
)

print("\nSample rows:")
display(df_events.head(10))

print("\nWarehouse dimension sample:")
display(df_dim.head(10))

Columns:
['event_id', 'tracking_number', 'event_ts', 'country', 'device', 'package_weight_kg', 'delivery_cost', 'tags', 'event_date', 'status', 'delay_minutes', 'warehouse_id']

Status distribution:


status,count,percent
str,u32,f64
"""in_transit""",4001431,40.01
"""delivered""",2998974,29.99
"""out_for_delivery""",1499963,15.0
"""picked_up""",998709,9.99
"""delayed""",500923,5.01



Null counts:


column,null_count
str,u32
"""event_id""",0
"""tracking_number""",0
"""event_ts""",0
"""country""",0
"""device""",0
…,…
"""tags""",0
"""event_date""",0
"""status""",0



Sample rows:


event_id,tracking_number,event_ts,country,device,package_weight_kg,delivery_cost,tags,event_date,status,delay_minutes,warehouse_id
i64,i64,datetime[μs],str,str,f64,i64,list[str],date,str,i64,i64
1,14676,2026-02-02 14:10:03,"""FR""","""mobile""",85.653,2839,"[""security"", ""polars"", ""cloud""]",2026-02-02,"""out_for_delivery""",0,158
2,2880,2026-01-07 19:27:08,"""IN""","""desktop""",67.111,8516,"[""duckdb"", ""spark"", ""polars""]",2026-01-07,"""in_transit""",0,760
3,3134,2026-03-16 00:38:56,"""IN""","""mobile""",18.635,5126,"[""spark"", ""security"", ""cloud""]",2026-03-16,"""picked_up""",0,440
4,165256,2026-01-30 23:23:46,"""IN""","""tablet""",21.947,2199,"[""duckdb"", ""spark""]",2026-01-30,"""out_for_delivery""",0,781
5,103798,2026-01-02 11:12:27,"""IN""","""mobile""",76.217,5244,"[""etl""]",2026-01-02,"""out_for_delivery""",0,935
6,154410,2026-01-06 05:20:33,"""FR""","""mobile""",211.618,4169,"[""mlops"", ""security""]",2026-01-06,"""in_transit""",0,3
7,3571,2026-03-22 18:15:28,"""UK""","""mobile""",60.532,1397,"[""ai"", ""sql"", ""etl""]",2026-03-22,"""in_transit""",0,564
8,70444,2026-03-02 09:06:31,"""DE""","""mobile""",29.373,1420,"[""mlops"", ""duckdb""]",2026-03-02,"""delivered""",0,339
9,1278,2026-02-18 14:47:40,"""US""","""mobile""",16.229,1677,"[""etl"", ""ai"", ""etl""]",2026-02-18,"""delayed""",535,187



Warehouse dimension sample:


warehouse_id,warehouse_region,warehouse_capacity
i64,str,i64
1,"""North""",23614
2,"""Central""",12928
3,"""South""",5246
4,"""West""",48928
5,"""East""",19001
6,"""West""",5805
7,"""East""",6789
8,"""Central""",34102
9,"""East""",19774


## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [172]:
BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []

def parquet_size_mb(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size / (1024 * 1024)
    if path.is_dir():
        return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / (1024 * 1024)
    return 0.0


def normalize_result(result):
    """
    Convert different result objects to a comparable lightweight representation.
    We do not need full equality of DataFrame internals, only stable output shape
    and basic values for sanity checks.
    """
    if isinstance(result, pl.DataFrame):
        return {
            "type": "polars",
            "shape": result.shape,
            "columns": result.columns,
        }

    if isinstance(result, pd.DataFrame):
        return {
            "type": "pandas",
            "shape": result.shape,
            "columns": list(result.columns),
        }

    try:
        # Spark DataFrame
        if result.__class__.__name__ == "DataFrame":
            return {
                "type": "spark",
                "shape": (result.count(), len(result.columns)),
                "columns": result.columns,
            }
    except Exception:
        pass

    return {
        "type": type(result).__name__,
        "repr": str(result)[:200],
    }


def run_benchmark(
    query_name,
    engine,
    mode,
    func,
    repetitions=3,
    data_format="parquet",
    layout="default",
    input_path=EVENTS_PATH,
    result_check="manual",
    notes="",
):
    times = []
    peak_memories = []
    last_result = None

    for _ in range(repetitions):
        gc.collect()

        start_time = time.perf_counter()
        mem_usage, result = memory_usage(
            (func, (), {}),
            retval=True,
            max_usage=True,
            interval=0.1,
        )
        end_time = time.perf_counter()

        times.append(end_time - start_time)

        peak_mem = mem_usage[0] if isinstance(mem_usage, list) else mem_usage
        peak_memories.append(float(peak_mem))

        last_result = result

    result_summary = normalize_result(last_result)

    result = {
        "library_engine": engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": N_ROWS,
        "median_time_s": round(float(np.median(times)), 4),
        "peak_memory_mb": round(float(max(peak_memories)), 2),
        "input_size_mb": round(float(parquet_size_mb(input_path)), 2),
        "result_check": result_check,
        "notes": notes + f" | result={result_summary}",
    }

    benchmark_results.append(result)

    print(
        f"-> [{engine} | {mode}] {query_name}: "
        f"{result['median_time_s']} s, peak RSS {result['peak_memory_mb']} MB"
    )

    return result

## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


### Benchmark query design

#### Q1: Delayed deliveries by warehouse region

This query filters events with `status = "delayed"`, joins the fact table with the warehouse dimension table on `warehouse_id`, groups by `warehouse_region`, and computes the number of delayed deliveries, average delay, maximum delay, and average delivery cost.

This query tests selective filtering, join with a dimension table, and group-by aggregation. I expect DuckDB and Polars lazy to perform well because both can push filters and projections into the Parquet scan. Pandas may use more memory because it usually materializes the input DataFrame before filtering. The optimized Parquet layout sorted by `event_date` and `status` may help only partially, because this query filters by `status` but not by a narrow date range.

#### Q2: Top expensive in-transit deliveries

This query filters events with `status = "in_transit"`, selects only a few relevant columns, sorts by `delivery_cost` in descending order, and returns the top 100 records.

This query tests top-k sorting and column pruning. I expect Polars lazy and DuckDB to perform well because they can avoid reading unnecessary columns and optimize sorting/top-k execution. Pandas may be slower and more memory-intensive because it reads the full DataFrame first. Physical layout should not help much unless the engine can exploit column pruning from Parquet.

#### Q3: Daily delivered package weight

This query filters events with `status = "delivered"` and a selected event date range, groups by `event_date`, and computes the number of delivered packages and the total and average package weight.

This query tests date filtering, predicate pushdown, partition pruning, and aggregation. I expect DuckDB and Polars lazy to benefit from Parquet predicate pushdown, especially on the partitioned layout by `event_date`. Pandas may use the most memory because it loads the whole file before filtering unless manually optimized. The partitioned layout should help this query the most.

In [173]:
QUERY_SPECS = [
    {
        "query_name": "Q1_delayed_by_region",
        "description": "Delayed deliveries joined with warehouse dimension and aggregated by warehouse region.",
        "classes": [
            "selective filter plus aggregation",
            "join with a dimension table",
            "group-by aggregation",
        ],
        "expected_best": "DuckDB or Polars lazy",
        "expected_memory_heaviest": "Pandas",
        "layout_expected_to_help": "optimized layout may help partially because data is sorted by status and event_date",
    },
    {
        "query_name": "Q2_top_expensive_in_transit",
        "description": "Top 100 in-transit deliveries sorted by delivery_cost.",
        "classes": [
            "top-k or sorting",
            "column pruning",
            "selective filter",
        ],
        "expected_best": "Polars lazy or DuckDB",
        "expected_memory_heaviest": "Pandas",
        "layout_expected_to_help": "column pruning should help more than row layout",
    },
    {
        "query_name": "Q3_daily_delivered_weight",
        "description": "Delivered packages in a date range aggregated by event_date.",
        "classes": [
            "query sensitive to partitioned vs unpartitioned layout",
            "predicate pushdown",
            "group-by aggregation",
        ],
        "expected_best": "DuckDB or Polars lazy on partitioned Parquet",
        "expected_memory_heaviest": "Pandas",
        "layout_expected_to_help": "partitioned layout by event_date should help significantly",
    },
]

pd.DataFrame(QUERY_SPECS)

,query_name,description,classes,expected_best,expected_memory_heaviest,layout_expected_to_help
0,Q1_delayed_by_region,Delayed deliveries joined with warehouse dimen...,"[selective filter plus aggregation, join with ...",DuckDB or Polars lazy,Pandas,optimized layout may help partially because da...
1,Q2_top_expensive_in_transit,Top 100 in-transit deliveries sorted by delive...,"[top-k or sorting, column pruning, selective f...",Polars lazy or DuckDB,Pandas,column pruning should help more than row layout
2,Q3_daily_delivered_weight,Delivered packages in a date range aggregated ...,[query sensitive to partitioned vs unpartition...,DuckDB or Polars lazy on partitioned Parquet,Pandas,partitioned layout by event_date should help s...


### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


### Pandas Default Implementations

In [174]:
DATE_FROM = pd.Timestamp("2026-02-01").date()
DATE_TO = pd.Timestamp("2026-02-15").date()


def pandas_default_q1_delayed_by_region():
    events = pd.read_parquet(EVENTS_PATH)
    dim = pd.read_parquet(DIMENSION_PATH)

    delayed = events[events["status"] == "delayed"]

    result = (
        delayed
        .merge(dim, on="warehouse_id", how="inner")
        .groupby("warehouse_region", as_index=False)
        .agg(
            delayed_count=("event_id", "count"),
            avg_delay_minutes=("delay_minutes", "mean"),
            max_delay_minutes=("delay_minutes", "max"),
            avg_delivery_cost=("delivery_cost", "mean"),
        )
        .sort_values("delayed_count", ascending=False)
    )

    return result


def pandas_default_q2_top_expensive_in_transit():
    events = pd.read_parquet(EVENTS_PATH)

    result = (
        events.loc[
            events["status"] == "in_transit",
            ["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"],
        ]
        .sort_values(["delivery_cost", "event_id"], ascending=[False, True])
        .head(100)
    )

    return result


def pandas_default_q3_daily_delivered_weight():
    events = pd.read_parquet(EVENTS_PATH)

    mask = (
        (events["status"] == "delivered")
        & (events["event_date"] >= DATE_FROM)
        & (events["event_date"] <= DATE_TO)
    )

    result = (
        events.loc[mask]
        .groupby("event_date", as_index=False)
        .agg(
            delivered_count=("event_id", "count"),
            total_package_weight_kg=("package_weight_kg", "sum"),
            avg_package_weight_kg=("package_weight_kg", "mean"),
        )
        .sort_values("event_date")
    )

    return result

In [175]:
display(pandas_default_q1_delayed_by_region())
display(pandas_default_q2_top_expensive_in_transit().head())
display(pandas_default_q3_daily_delivered_weight())

,warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
0,Central,110494,725.630695,1439,4994.373713
3,South,103178,727.417802,1439,5000.420506
1,East,99861,726.066382,1439,4994.377224
2,North,99735,725.296135,1439,5021.068873
4,West,87655,730.110501,1439,4995.077588


,event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
16566,16567,163780,2026-02-15 09:14:29,US,9999,417
21827,21828,3017,2026-01-17 14:12:45,DE,9999,925
45685,45686,174408,2026-01-25 16:39:57,IN,9999,305
52133,52134,170251,2026-03-20 02:28:14,UK,9999,806
52299,52300,1885,2026-03-14 12:15:08,BR,9999,987


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,33623,2962591.389,88.112048
1,2026-02-02,33152,2963681.762,89.396771
2,2026-02-03,33320,2985405.880,89.598016
3,2026-02-04,33529,3056911.942,91.172178
4,2026-02-05,33345,3001674.820,90.018738
5,2026-02-06,33335,3010572.170,90.312649
6,2026-02-07,33278,2975862.899,89.424331
7,2026-02-08,33123,2980307.062,89.976967
8,2026-02-09,33062,2956250.754,89.415364
9,2026-02-10,33250,3004073.816,90.348085


### Pandas PyArrow Backend Implementations

In [176]:
def pandas_pyarrow_q1_delayed_by_region():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    dim = pd.read_parquet(DIMENSION_PATH, engine="pyarrow", dtype_backend="pyarrow")

    delayed = events[events["status"] == "delayed"]

    result = (
        delayed
        .merge(dim, on="warehouse_id", how="inner")
        .groupby("warehouse_region", as_index=False)
        .agg(
            delayed_count=("event_id", "count"),
            avg_delay_minutes=("delay_minutes", "mean"),
            max_delay_minutes=("delay_minutes", "max"),
            avg_delivery_cost=("delivery_cost", "mean"),
        )
        .sort_values("delayed_count", ascending=False)
    )

    return result


def pandas_pyarrow_q2_top_expensive_in_transit():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")

    result = (
        events.loc[
            events["status"] == "in_transit",
            ["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"],
        ]
        .sort_values(["delivery_cost", "event_id"], ascending=[False, True])
        .head(100)
    )

    return result


def pandas_pyarrow_q3_daily_delivered_weight():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")

    mask = (
        (events["status"] == "delivered")
        & (events["event_date"] >= DATE_FROM)
        & (events["event_date"] <= DATE_TO)
    )

    result = (
        events.loc[mask]
        .groupby("event_date", as_index=False)
        .agg(
            delivered_count=("event_id", "count"),
            total_package_weight_kg=("package_weight_kg", "sum"),
            avg_package_weight_kg=("package_weight_kg", "mean"),
        )
        .sort_values("event_date")
    )

    return result

In [177]:
display(pandas_pyarrow_q1_delayed_by_region())
display(pandas_pyarrow_q2_top_expensive_in_transit().head())
display(pandas_pyarrow_q3_daily_delivered_weight())

,warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
0,Central,110494,725.630695,1439,4994.373713
3,South,103178,727.417802,1439,5000.420506
1,East,99861,726.066382,1439,4994.377224
2,North,99735,725.296135,1439,5021.068873
4,West,87655,730.110501,1439,4995.077588


,event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
16566,16567,163780,2026-02-15 09:14:29,US,9999,417
21827,21828,3017,2026-01-17 14:12:45,DE,9999,925
45685,45686,174408,2026-01-25 16:39:57,IN,9999,305
52133,52134,170251,2026-03-20 02:28:14,UK,9999,806
52299,52300,1885,2026-03-14 12:15:08,BR,9999,987


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,33623,2962591.389,88.112048
1,2026-02-02,33152,2963681.762,89.396771
2,2026-02-03,33320,2985405.88,89.598016
3,2026-02-04,33529,3056911.942,91.172178
4,2026-02-05,33345,3001674.82,90.018738
5,2026-02-06,33335,3010572.17,90.312649
6,2026-02-07,33278,2975862.899,89.424331
7,2026-02-08,33123,2980307.062,89.976967
8,2026-02-09,33062,2956250.754,89.415364
9,2026-02-10,33250,3004073.816,90.348085


### Polars Eager Implementations

In [178]:
def polars_eager_q1_delayed_by_region():
    events = pl.read_parquet(EVENTS_PATH)
    dim = pl.read_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "delayed")
        .join(dim, on="warehouse_id", how="inner")
        .group_by("warehouse_region")
        .agg(
            pl.len().alias("delayed_count"),
            pl.col("delay_minutes").mean().alias("avg_delay_minutes"),
            pl.col("delay_minutes").max().alias("max_delay_minutes"),
            pl.col("delivery_cost").mean().alias("avg_delivery_cost"),
        )
        .sort("delayed_count", descending=True)
    )

    return result


def polars_eager_q2_top_expensive_in_transit():
    events = pl.read_parquet(EVENTS_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .select(["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"])
        .sort(["delivery_cost", "event_id"], descending=[True, False])
        .head(100)
    )

    return result


def polars_eager_q3_daily_delivered_weight():
    events = pl.read_parquet(EVENTS_PATH)

    result = (
        events
        .filter(
            (pl.col("status") == "delivered")
            & (pl.col("event_date") >= DATE_FROM)
            & (pl.col("event_date") <= DATE_TO)
        )
        .group_by("event_date")
        .agg(
            pl.len().alias("delivered_count"),
            pl.col("package_weight_kg").sum().alias("total_package_weight_kg"),
            pl.col("package_weight_kg").mean().alias("avg_package_weight_kg"),
        )
        .sort("event_date")
    )

    return result

In [179]:
display(polars_eager_q1_delayed_by_region())
display(polars_eager_q2_top_expensive_in_transit().head())
display(polars_eager_q3_daily_delivered_weight())

warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
str,u32,f64,i64,f64
"""Central""",110494,725.630695,1439,4994.373713
"""South""",103178,727.417802,1439,5000.420506
"""East""",99861,726.066382,1439,4994.377224
"""North""",99735,725.296135,1439,5021.068873
"""West""",87655,730.110501,1439,4995.077588


event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
i64,i64,datetime[μs],str,i64,i64
16567,163780,2026-02-15 09:14:29,"""US""",9999,417
21828,3017,2026-01-17 14:12:45,"""DE""",9999,925
45686,174408,2026-01-25 16:39:57,"""IN""",9999,305
52134,170251,2026-03-20 02:28:14,"""UK""",9999,806
52300,1885,2026-03-14 12:15:08,"""BR""",9999,987


event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
date,u32,f64,f64
2026-02-01,33623,2.9626e6,88.112048
2026-02-02,33152,2.9637e6,89.396771
2026-02-03,33320,2.9854e6,89.598016
2026-02-04,33529,3.0569e6,91.172178
2026-02-05,33345,3.0017e6,90.018738
…,…,…,…
2026-02-11,32997,2.9397e6,89.090829
2026-02-12,33545,3.0212e6,90.062889
2026-02-13,33176,2.9555e6,89.08687


### Polars Lazy Implementations

In [180]:
def polars_lazy_q1_delayed_by_region():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "delayed")
        .join(dim, on="warehouse_id", how="inner")
        .group_by("warehouse_region")
        .agg(
            pl.len().alias("delayed_count"),
            pl.col("delay_minutes").mean().alias("avg_delay_minutes"),
            pl.col("delay_minutes").max().alias("max_delay_minutes"),
            pl.col("delivery_cost").mean().alias("avg_delivery_cost"),
        )
        .sort("delayed_count", descending=True)
        .collect()
    )

    return result


def polars_lazy_q2_top_expensive_in_transit():
    events = pl.scan_parquet(EVENTS_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .select(["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"])
        .sort(["delivery_cost", "event_id"], descending=[True, False])
        .head(100)
        .collect()
    )

    return result


def polars_lazy_q3_daily_delivered_weight():
    events = pl.scan_parquet(EVENTS_PATH)

    result = (
        events
        .filter(
            (pl.col("status") == "delivered")
            & (pl.col("event_date") >= DATE_FROM)
            & (pl.col("event_date") <= DATE_TO)
        )
        .group_by("event_date")
        .agg(
            pl.len().alias("delivered_count"),
            pl.col("package_weight_kg").sum().alias("total_package_weight_kg"),
            pl.col("package_weight_kg").mean().alias("avg_package_weight_kg"),
        )
        .sort("event_date")
        .collect()
    )

    return result

In [181]:
display(polars_lazy_q1_delayed_by_region())
display(polars_lazy_q2_top_expensive_in_transit().head())
display(polars_lazy_q3_daily_delivered_weight())

warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
str,u32,f64,i64,f64
"""Central""",110494,725.630695,1439,4994.373713
"""South""",103178,727.417802,1439,5000.420506
"""East""",99861,726.066382,1439,4994.377224
"""North""",99735,725.296135,1439,5021.068873
"""West""",87655,730.110501,1439,4995.077588


event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
i64,i64,datetime[μs],str,i64,i64
16567,163780,2026-02-15 09:14:29,"""US""",9999,417
21828,3017,2026-01-17 14:12:45,"""DE""",9999,925
45686,174408,2026-01-25 16:39:57,"""IN""",9999,305
52134,170251,2026-03-20 02:28:14,"""UK""",9999,806
52300,1885,2026-03-14 12:15:08,"""BR""",9999,987


event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
date,u32,f64,f64
2026-02-01,33623,2.9626e6,88.112048
2026-02-02,33152,2.9637e6,89.396771
2026-02-03,33320,2.9854e6,89.598016
2026-02-04,33529,3.0569e6,91.172178
2026-02-05,33345,3.0017e6,90.018738
…,…,…,…
2026-02-11,32997,2.9397e6,89.090829
2026-02-12,33545,3.0212e6,90.062889
2026-02-13,33176,2.9555e6,89.08687


### Polars Lazy Streaming Implementations

In [182]:
def polars_streaming_q1_delayed_by_region():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "delayed")
        .join(dim, on="warehouse_id", how="inner")
        .group_by("warehouse_region")
        .agg(
            pl.len().alias("delayed_count"),
            pl.col("delay_minutes").mean().alias("avg_delay_minutes"),
            pl.col("delay_minutes").max().alias("max_delay_minutes"),
            pl.col("delivery_cost").mean().alias("avg_delivery_cost"),
        )
        .sort("delayed_count", descending=True)
        .collect(engine="streaming")
    )

    return result


def polars_streaming_q2_top_expensive_in_transit():
    events = pl.scan_parquet(EVENTS_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .select(["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"])
        .sort(["delivery_cost", "event_id"], descending=[True, False])
        .head(100)
        .collect(engine="streaming")
    )

    return result


def polars_streaming_q3_daily_delivered_weight():
    events = pl.scan_parquet(EVENTS_PATH)

    result = (
        events
        .filter(
            (pl.col("status") == "delivered")
            & (pl.col("event_date") >= DATE_FROM)
            & (pl.col("event_date") <= DATE_TO)
        )
        .group_by("event_date")
        .agg(
            pl.len().alias("delivered_count"),
            pl.col("package_weight_kg").sum().alias("total_package_weight_kg"),
            pl.col("package_weight_kg").mean().alias("avg_package_weight_kg"),
        )
        .sort("event_date")
        .collect(engine="streaming")
    )

    return result

In [183]:
display(polars_streaming_q1_delayed_by_region())
display(polars_streaming_q2_top_expensive_in_transit().head())
display(polars_streaming_q3_daily_delivered_weight())

warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
str,u32,f64,i64,f64
"""Central""",110494,725.630695,1439,4994.373713
"""South""",103178,727.417802,1439,5000.420506
"""East""",99861,726.066382,1439,4994.377224
"""North""",99735,725.296135,1439,5021.068873
"""West""",87655,730.110501,1439,4995.077588


event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
i64,i64,datetime[μs],str,i64,i64
16567,163780,2026-02-15 09:14:29,"""US""",9999,417
21828,3017,2026-01-17 14:12:45,"""DE""",9999,925
45686,174408,2026-01-25 16:39:57,"""IN""",9999,305
52134,170251,2026-03-20 02:28:14,"""UK""",9999,806
52300,1885,2026-03-14 12:15:08,"""BR""",9999,987


event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
date,u32,f64,f64
2026-02-01,33623,2.9626e6,88.112048
2026-02-02,33152,2.9637e6,89.396771
2026-02-03,33320,2.9854e6,89.598016
2026-02-04,33529,3.0569e6,91.172178
2026-02-05,33345,3.0017e6,90.018738
…,…,…,…
2026-02-11,32997,2.9397e6,89.090829
2026-02-12,33545,3.0212e6,90.062889
2026-02-13,33176,2.9555e6,89.08687


### DuckDB Implementations

In [184]:
def duckdb_q1_delayed_by_region():
    con = duckdb.connect(database=":memory:")

    result = con.execute(
        """
        SELECT
            d.warehouse_region,
            COUNT(*) AS delayed_count,
            AVG(e.delay_minutes) AS avg_delay_minutes,
            MAX(e.delay_minutes) AS max_delay_minutes,
            AVG(e.delivery_cost) AS avg_delivery_cost
        FROM read_parquet(?) AS e
        INNER JOIN read_parquet(?) AS d
            ON e.warehouse_id = d.warehouse_id
        WHERE e.status = 'delayed'
        GROUP BY d.warehouse_region
        ORDER BY delayed_count DESC
        """,
        [str(EVENTS_PATH), str(DIMENSION_PATH)],
    ).fetchdf()

    con.close()
    return result


def duckdb_q2_top_expensive_in_transit():
    con = duckdb.connect(database=":memory:")

    result = con.execute(
        """
        SELECT
            event_id,
            tracking_number,
            event_ts,
            country,
            delivery_cost,
            warehouse_id
        FROM read_parquet(?)
        WHERE status = 'in_transit'
        ORDER BY delivery_cost DESC, event_id ASC
        LIMIT 100
        """,
        [str(EVENTS_PATH)],
    ).fetchdf()

    con.close()
    return result


def duckdb_q3_daily_delivered_weight():
    con = duckdb.connect(database=":memory:")

    result = con.execute(
        """
        SELECT
            event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_parquet(?)
        WHERE status = 'delivered'
          AND event_date BETWEEN ? AND ?
        GROUP BY event_date
        ORDER BY event_date
        """,
        [str(EVENTS_PATH), DATE_FROM, DATE_TO],
    ).fetchdf()

    con.close()
    return result

In [185]:
display(duckdb_q1_delayed_by_region())
display(duckdb_q2_top_expensive_in_transit().head())
display(duckdb_q3_daily_delivered_weight())

,warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
0,Central,110494,725.630695,1439,4994.373713
1,South,103178,727.417802,1439,5000.420506
2,East,99861,726.066382,1439,4994.377224
3,North,99735,725.296135,1439,5021.068873
4,West,87655,730.110501,1439,4995.077588


,event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
0,16567,163780,2026-02-15 09:14:29,US,9999,417
1,21828,3017,2026-01-17 14:12:45,DE,9999,925
2,45686,174408,2026-01-25 16:39:57,IN,9999,305
3,52134,170251,2026-03-20 02:28:14,UK,9999,806
4,52300,1885,2026-03-14 12:15:08,BR,9999,987


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,33623,2962591.389,88.112048
1,2026-02-02,33152,2963681.762,89.396771
2,2026-02-03,33320,2985405.880,89.598016
3,2026-02-04,33529,3056911.942,91.172178
4,2026-02-05,33345,3001674.820,90.018738
5,2026-02-06,33335,3010572.170,90.312649
6,2026-02-07,33278,2975862.899,89.424331
7,2026-02-08,33123,2980307.062,89.976967
8,2026-02-09,33062,2956250.754,89.415364
9,2026-02-10,33250,3004073.816,90.348085


In [186]:
# TODO: // DONE
# Configure Spark local only when you start the PySpark local benchmark.
# Initialize Spark only when you start the Spark part of the benchmark.
# TODO:// DONE
# Adjust memory and local core count if needed.

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

### PySpark Local Implementations

In [187]:
from pyspark.sql import functions as F

def spark_q1_delayed_by_region():
    events = spark.read.parquet(str(EVENTS_PATH))
    dim = spark.read.parquet(str(DIMENSION_PATH))

    result = (
        events
        .filter(F.col("status") == "delayed")
        .join(dim, on="warehouse_id", how="inner")
        .groupBy("warehouse_region")
        .agg(
            F.count("*").alias("delayed_count"),
            F.avg("delay_minutes").alias("avg_delay_minutes"),
            F.max("delay_minutes").alias("max_delay_minutes"),
            F.avg("delivery_cost").alias("avg_delivery_cost"),
        )
        .orderBy(F.col("delayed_count").desc())
    )

    return result.toPandas()


def spark_q2_top_expensive_in_transit():
    events = spark.read.parquet(str(EVENTS_PATH))

    result = (
        events
        .filter(F.col("status") == "in_transit")
        .select("event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id")
        .orderBy(F.col("delivery_cost").desc(), F.col("event_id").asc())
        .limit(100)
    )

    return result.toPandas()


def spark_q3_daily_delivered_weight():
    events = spark.read.parquet(str(EVENTS_PATH))

    result = (
        events
        .filter(
            (F.col("status") == "delivered")
            & (F.col("event_date") >= F.lit(str(DATE_FROM)))
            & (F.col("event_date") <= F.lit(str(DATE_TO)))
        )
        .groupBy("event_date")
        .agg(
            F.count("*").alias("delivered_count"),
            F.sum("package_weight_kg").alias("total_package_weight_kg"),
            F.avg("package_weight_kg").alias("avg_package_weight_kg"),
        )
        .orderBy("event_date")
    )

    return result.toPandas()

In [188]:
display(spark_q1_delayed_by_region())
display(spark_q2_top_expensive_in_transit().head())
display(spark_q3_daily_delivered_weight())

,warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
0,Central,110494,725.630695,1439,4994.373713
1,South,103178,727.417802,1439,5000.420506
2,East,99861,726.066382,1439,4994.377224
3,North,99735,725.296135,1439,5021.068873
4,West,87655,730.110501,1439,4995.077588


,event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
0,16567,163780,2026-02-15 09:14:29,US,9999,417
1,21828,3017,2026-01-17 14:12:45,DE,9999,925
2,45686,174408,2026-01-25 16:39:57,IN,9999,305
3,52134,170251,2026-03-20 02:28:14,UK,9999,806
4,52300,1885,2026-03-14 12:15:08,BR,9999,987


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,33623,2962591.389,88.112048
1,2026-02-02,33152,2963681.762,89.396771
2,2026-02-03,33320,2985405.880,89.598016
3,2026-02-04,33529,3056911.942,91.172178
4,2026-02-05,33345,3001674.820,90.018738
5,2026-02-06,33335,3010572.170,90.312649
6,2026-02-07,33278,2975862.899,89.424331
7,2026-02-08,33123,2980307.062,89.976967
8,2026-02-09,33062,2956250.754,89.415364
9,2026-02-10,33250,3004073.816,90.348085


### Run local benchmark suite

In [189]:
benchmark_results = []

LOCAL_BENCHMARKS = [
    # Pandas default
    ("Q1_delayed_by_region", "Pandas", "default_numpy", pandas_default_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Pandas", "default_numpy", pandas_default_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Pandas", "default_numpy", pandas_default_q3_daily_delivered_weight),

    # Pandas PyArrow
    ("Q1_delayed_by_region", "Pandas", "pyarrow_backend", pandas_pyarrow_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Pandas", "pyarrow_backend", pandas_pyarrow_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Pandas", "pyarrow_backend", pandas_pyarrow_q3_daily_delivered_weight),

    # Polars eager
    ("Q1_delayed_by_region", "Polars", "eager", polars_eager_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Polars", "eager", polars_eager_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Polars", "eager", polars_eager_q3_daily_delivered_weight),

    # Polars lazy
    ("Q1_delayed_by_region", "Polars", "lazy_collect", polars_lazy_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Polars", "lazy_collect", polars_lazy_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Polars", "lazy_collect", polars_lazy_q3_daily_delivered_weight),

    # Polars streaming
    ("Q1_delayed_by_region", "Polars", "lazy_streaming", polars_streaming_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Polars", "lazy_streaming", polars_streaming_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Polars", "lazy_streaming", polars_streaming_q3_daily_delivered_weight),

    # DuckDB
    ("Q1_delayed_by_region", "DuckDB", "sql_read_parquet", duckdb_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "DuckDB", "sql_read_parquet", duckdb_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "DuckDB", "sql_read_parquet", duckdb_q3_daily_delivered_weight),

    # PySpark local
    ("Q1_delayed_by_region", "PySpark", "local[*]", spark_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "PySpark", "local[*]", spark_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "PySpark", "local[*]", spark_q3_daily_delivered_weight),
]

for query_name, engine, mode, func in LOCAL_BENCHMARKS:
    run_benchmark(
        query_name=query_name,
        engine=engine,
        mode=mode,
        func=func,
        repetitions=3,
        data_format="parquet",
        layout="default",
        input_path=EVENTS_PATH,
        result_check="passed",
        notes="Local benchmark on the same generated Parquet dataset. Peak memory is process RSS measured from the notebook kernel.",
    )

benchmark_df = pd.DataFrame(benchmark_results, columns=BENCHMARK_COLUMNS)
display(benchmark_df)

-> [Pandas | default_numpy] Q1_delayed_by_region: 3.6353 s, peak RSS 10915.57 MB
-> [Pandas | default_numpy] Q2_top_expensive_in_transit: 4.2598 s, peak RSS 11706.75 MB
-> [Pandas | default_numpy] Q3_daily_delivered_weight: 4.2689 s, peak RSS 11497.01 MB
-> [Pandas | pyarrow_backend] Q1_delayed_by_region: 1.5594 s, peak RSS 10285.26 MB
-> [Pandas | pyarrow_backend] Q2_top_expensive_in_transit: 2.2409 s, peak RSS 9806.1 MB
-> [Pandas | pyarrow_backend] Q3_daily_delivered_weight: 1.5464 s, peak RSS 9567.36 MB
-> [Polars | eager] Q1_delayed_by_region: 0.8254 s, peak RSS 11246.86 MB
-> [Polars | eager] Q2_top_expensive_in_transit: 1.1692 s, peak RSS 11916.14 MB
-> [Polars | eager] Q3_daily_delivered_weight: 0.8891 s, peak RSS 11338.29 MB
-> [Polars | lazy_collect] Q1_delayed_by_region: 0.7137 s, peak RSS 9990.51 MB
-> [Polars | lazy_collect] Q2_top_expensive_in_transit: 0.7072 s, peak RSS 10049.39 MB
-> [Polars | lazy_collect] Q3_daily_delivered_weight: 0.6978 s, peak RSS 9447.61 MB
-> [Po

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Pandas,default_numpy,Q1_delayed_by_region,parquet,default,10000000,3.6353,10915.57,209.23,passed,Local benchmark on the same generated Parquet ...
1,Pandas,default_numpy,Q2_top_expensive_in_transit,parquet,default,10000000,4.2598,11706.75,209.23,passed,Local benchmark on the same generated Parquet ...
2,Pandas,default_numpy,Q3_daily_delivered_weight,parquet,default,10000000,4.2689,11497.01,209.23,passed,Local benchmark on the same generated Parquet ...
3,Pandas,pyarrow_backend,Q1_delayed_by_region,parquet,default,10000000,1.5594,10285.26,209.23,passed,Local benchmark on the same generated Parquet ...
4,Pandas,pyarrow_backend,Q2_top_expensive_in_transit,parquet,default,10000000,2.2409,9806.10,209.23,passed,Local benchmark on the same generated Parquet ...
5,Pandas,pyarrow_backend,Q3_daily_delivered_weight,parquet,default,10000000,1.5464,9567.36,209.23,passed,Local benchmark on the same generated Parquet ...
6,Polars,eager,Q1_delayed_by_region,parquet,default,10000000,0.8254,11246.86,209.23,passed,Local benchmark on the same generated Parquet ...
7,Polars,eager,Q2_top_expensive_in_transit,parquet,default,10000000,1.1692,11916.14,209.23,passed,Local benchmark on the same generated Parquet ...
8,Polars,eager,Q3_daily_delivered_weight,parquet,default,10000000,0.8891,11338.29,209.23,passed,Local benchmark on the same generated Parquet ...
9,Polars,lazy_collect,Q1_delayed_by_region,parquet,default,10000000,0.7137,9990.51,209.23,passed,Local benchmark on the same generated Parquet ...


In [190]:
RESULTS_PATH = OUTPUT_DIR / "local_benchmark_results.csv"
benchmark_df.to_csv(RESULTS_PATH, index=False)
RESULTS_PATH

PosixPath('../data/phase2_26L/group_11/local_benchmark_results.csv')

### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [191]:
# Prepare a flat CSV baseline for Q3. The original dataset contains a list column (`tags`),
# so the CSV baseline contains only columns needed by the selected query.

Q3_COLUMNS = ["event_id", "event_date", "status", "package_weight_kg"]

q3_flat = pl.read_parquet(EVENTS_PATH).select(Q3_COLUMNS)
q3_flat.write_csv(CSV_EVENTS_PATH)

print("Default Parquet size MB:", round(parquet_size_mb(EVENTS_PATH), 2))
print("Optimized Parquet size MB:", round(parquet_size_mb(OPTIMIZED_EVENTS_PATH), 2))
print("Partitioned Parquet size MB:", round(parquet_size_mb(PARTITIONED_EVENTS_DIR), 2))
print("CSV baseline size MB:", round(parquet_size_mb(CSV_EVENTS_PATH), 2))

Default Parquet size MB: 209.23
Optimized Parquet size MB: 214.29
Partitioned Parquet size MB: 196.51
CSV baseline size MB: 356.28


In [192]:
def duckdb_q3_default_parquet():
    con = duckdb.connect(database=":memory:")
    result = con.execute(
        """
        SELECT
            event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_parquet(?)
        WHERE status = 'delivered'
          AND event_date BETWEEN ? AND ?
        GROUP BY event_date
        ORDER BY event_date
        """,
        [str(EVENTS_PATH), DATE_FROM, DATE_TO],
    ).fetchdf()
    con.close()
    return result


def duckdb_q3_optimized_parquet():
    con = duckdb.connect(database=":memory:")
    result = con.execute(
        """
        SELECT
            event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_parquet(?)
        WHERE status = 'delivered'
          AND event_date BETWEEN ? AND ?
        GROUP BY event_date
        ORDER BY event_date
        """,
        [str(OPTIMIZED_EVENTS_PATH), DATE_FROM, DATE_TO],
    ).fetchdf()
    con.close()
    return result


def duckdb_q3_partitioned_parquet():
    con = duckdb.connect(database=":memory:")
    result = con.execute(
        """
        SELECT
            event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_parquet(?)
        WHERE status = 'delivered'
          AND event_date BETWEEN ? AND ?
        GROUP BY event_date
        ORDER BY event_date
        """,
        [str(PARTITIONED_EVENTS_DIR / "**" / "*.parquet"), DATE_FROM, DATE_TO],
    ).fetchdf()
    con.close()
    return result


def duckdb_q3_csv_baseline():
    con = duckdb.connect(database=":memory:")
    result = con.execute(
        """
        SELECT
            CAST(event_date AS DATE) AS event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_csv_auto(?)
        WHERE status = 'delivered'
          AND CAST(event_date AS DATE) BETWEEN ? AND ?
        GROUP BY CAST(event_date AS DATE)
        ORDER BY event_date
        """,
        [str(CSV_EVENTS_PATH), DATE_FROM, DATE_TO],
    ).fetchdf()
    con.close()
    return result

In [193]:
display(duckdb_q3_default_parquet())
display(duckdb_q3_optimized_parquet())
display(duckdb_q3_partitioned_parquet())
display(duckdb_q3_csv_baseline())

,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,33623,2962591.389,88.112048
1,2026-02-02,33152,2963681.762,89.396771
2,2026-02-03,33320,2985405.880,89.598016
3,2026-02-04,33529,3056911.942,91.172178
4,2026-02-05,33345,3001674.820,90.018738
5,2026-02-06,33335,3010572.170,90.312649
6,2026-02-07,33278,2975862.899,89.424331
7,2026-02-08,33123,2980307.062,89.976967
8,2026-02-09,33062,2956250.754,89.415364
9,2026-02-10,33250,3004073.816,90.348085


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,33623,2962591.389,88.112048
1,2026-02-02,33152,2963681.762,89.396771
2,2026-02-03,33320,2985405.880,89.598016
3,2026-02-04,33529,3056911.942,91.172178
4,2026-02-05,33345,3001674.820,90.018738
5,2026-02-06,33335,3010572.170,90.312649
6,2026-02-07,33278,2975862.899,89.424331
7,2026-02-08,33123,2980307.062,89.976967
8,2026-02-09,33062,2956250.754,89.415364
9,2026-02-10,33250,3004073.816,90.348085


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,33623,2962591.389,88.112048
1,2026-02-02,33152,2963681.762,89.396771
2,2026-02-03,33320,2985405.880,89.598016
3,2026-02-04,33529,3056911.942,91.172178
4,2026-02-05,33345,3001674.820,90.018738
5,2026-02-06,33335,3010572.170,90.312649
6,2026-02-07,33278,2975862.899,89.424331
7,2026-02-08,33123,2980307.062,89.976967
8,2026-02-09,33062,2956250.754,89.415364
9,2026-02-10,33250,3004073.816,90.348085


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,33623,2962591.389,88.112048
1,2026-02-02,33152,2963681.762,89.396771
2,2026-02-03,33320,2985405.880,89.598016
3,2026-02-04,33529,3056911.942,91.172178
4,2026-02-05,33345,3001674.820,90.018738
5,2026-02-06,33335,3010572.170,90.312649
6,2026-02-07,33278,2975862.899,89.424331
7,2026-02-08,33123,2980307.062,89.976967
8,2026-02-09,33062,2956250.754,89.415364
9,2026-02-10,33250,3004073.816,90.348085


In [194]:
q3_default = duckdb_q3_default_parquet()
q3_optimized = duckdb_q3_optimized_parquet()
q3_partitioned = duckdb_q3_partitioned_parquet()
q3_csv = duckdb_q3_csv_baseline()

pd.testing.assert_frame_equal(q3_default, q3_optimized, check_dtype=False)
pd.testing.assert_frame_equal(q3_default, q3_partitioned, check_dtype=False)
pd.testing.assert_frame_equal(q3_default, q3_csv, check_dtype=False)

print("Q3 layout equivalence check: passed")

Q3 layout equivalence check: passed


In [214]:
layout_benchmark_results = []

for query_name, engine, mode, func, data_format, layout, input_path in [
    ("Q3_daily_delivered_weight", "DuckDB", "default_parquet", duckdb_q3_default_parquet, "parquet", "default", EVENTS_PATH),
    ("Q3_daily_delivered_weight", "DuckDB", "optimized_parquet", duckdb_q3_optimized_parquet, "parquet", "optimized_sort_event_date_status", OPTIMIZED_EVENTS_PATH),
    ("Q3_daily_delivered_weight", "DuckDB", "partitioned_parquet", duckdb_q3_partitioned_parquet, "parquet", "partitioned_by_event_date", PARTITIONED_EVENTS_DIR),
    ("Q3_daily_delivered_weight", "DuckDB", "csv_baseline", duckdb_q3_csv_baseline, "csv", "flat_selected_columns", CSV_EVENTS_PATH),
]:
    before = len(benchmark_results)
    run_benchmark(
        query_name=query_name,
        engine=engine,
        mode=mode,
        func=func,
        repetitions=3,
        data_format=data_format,
        layout=layout,
        input_path=input_path,
        result_check="passed",
        notes="Task 2.5 layout/format comparison for Q3. DuckDB reads files directly.",
    )
    layout_benchmark_results.append(benchmark_results[-1])

layout_benchmark_df = pd.DataFrame(layout_benchmark_results, columns=BENCHMARK_COLUMNS)
display(layout_benchmark_df)

LAYOUT_RESULTS_PATH = OUTPUT_DIR / "layout_benchmark_results.csv"
layout_benchmark_df.to_csv(LAYOUT_RESULTS_PATH, index=False)
LAYOUT_RESULTS_PATH

-> [DuckDB | default_parquet] Q3_daily_delivered_weight: 0.7749 s, peak RSS 9996.04 MB
-> [DuckDB | optimized_parquet] Q3_daily_delivered_weight: 0.7279 s, peak RSS 9952.78 MB
-> [DuckDB | partitioned_parquet] Q3_daily_delivered_weight: 0.7558 s, peak RSS 9972.97 MB
-> [DuckDB | csv_baseline] Q3_daily_delivered_weight: 0.6894 s, peak RSS 10261.14 MB


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,DuckDB,default_parquet,Q3_daily_delivered_weight,parquet,default,10000000,0.7749,9996.04,209.23,passed,Task 2.5 layout/format comparison for Q3. Duck...
1,DuckDB,optimized_parquet,Q3_daily_delivered_weight,parquet,optimized_sort_event_date_status,10000000,0.7279,9952.78,214.29,passed,Task 2.5 layout/format comparison for Q3. Duck...
2,DuckDB,partitioned_parquet,Q3_daily_delivered_weight,parquet,partitioned_by_event_date,10000000,0.7558,9972.97,196.51,passed,Task 2.5 layout/format comparison for Q3. Duck...
3,DuckDB,csv_baseline,Q3_daily_delivered_weight,csv,flat_selected_columns,10000000,0.6894,10261.14,356.28,passed,Task 2.5 layout/format comparison for Q3. Duck...


PosixPath('../data/phase2_26L/group_11/layout_benchmark_results.csv')

### Task 2.5 analysis

For the file format and layout experiment I selected `Q3_daily_delivered_weight`, because this query filters by `status` and by a narrow `event_date` range and then aggregates by `event_date`. This makes it suitable for testing predicate pushdown and partition pruning.

The compared layouts were:

- default Parquet: single default Parquet file,
- optimized Parquet: data sorted by `event_date` and `status` with smaller row groups,
- partitioned Parquet: data partitioned by `event_date`,
- CSV baseline: flat CSV file containing only the columns needed by Q3.

The default Parquet file size was 209.23 MB, the optimized Parquet file size was 214.29 MB, the partitioned Parquet layout size was 196.51 MB, and the CSV baseline size was 356.28 MB. Even though the CSV baseline contained only selected columns, it was still larger than Parquet because CSV does not use columnar binary encoding and compression as efficiently as Parquet.

The fastest layout in terms of execution time was surprisingly the CSV baseline, with a runtime of 0.6894 s. However, it consumed the highest amount of memory (peak RSS 10261.14 MB). This highlights that while DuckDB has an exceptionally fast and optimized CSV reader, reading CSVs lacks the memory efficiency, column pruning, and typed storage provided by Parquet.

Among the Parquet formats, the optimized Parquet layout performed the best, with a runtime of 0.7279 s and the lowest memory footprint overall (peak RSS 9952.78 MB). This suggests that sorting by event_date and status (and adjusting row group sizes) successfully helped DuckDB minimize the data scanned for this particular query.

The partitioned Parquet layout was slightly slower at 0.7558 s (peak RSS 9972.97 MB). This indicates that while filtering by event_date benefits from partition pruning, managing multiple partitioned files might have introduced a slight overhead for DuckDB compared to reading a single optimized file. The default Parquet layout was the slowest overall at 0.7749 s.

All four variants returned equivalent results for the selected query. The memory measurements should be interpreted as approximate peak RSS values of the notebook process, not as exact memory used only by the query itself.

### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [215]:
STREAMING_OUTPUT_PATH = OUTPUT_DIR / "polars_streaming_large_output.parquet"

# Remove previous output if it exists, so every run writes a fresh file.
if STREAMING_OUTPUT_PATH.exists():
    STREAMING_OUTPUT_PATH.unlink()


def polars_eager_large_output():
    events = pl.read_parquet(EVENTS_PATH)
    dim = pl.read_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .join(dim, on="warehouse_id", how="inner")
        .select([
            "event_id",
            "tracking_number",
            "event_ts",
            "event_date",
            "country",
            "device",
            "delivery_cost",
            "package_weight_kg",
            "warehouse_id",
            "warehouse_region",
            "warehouse_capacity",
        ])
        .with_columns(
            (pl.col("delivery_cost") * 1.2).alias("projected_cost")
        )
    )

    return result


def polars_lazy_large_output():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .join(dim, on="warehouse_id", how="inner")
        .select([
            "event_id",
            "tracking_number",
            "event_ts",
            "event_date",
            "country",
            "device",
            "delivery_cost",
            "package_weight_kg",
            "warehouse_id",
            "warehouse_region",
            "warehouse_capacity",
        ])
        .with_columns(
            (pl.col("delivery_cost") * 1.2).alias("projected_cost")
        )
        .collect()
    )

    return result


def polars_streaming_collect_large_output():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .join(dim, on="warehouse_id", how="inner")
        .select([
            "event_id",
            "tracking_number",
            "event_ts",
            "event_date",
            "country",
            "device",
            "delivery_cost",
            "package_weight_kg",
            "warehouse_id",
            "warehouse_region",
            "warehouse_capacity",
        ])
        .with_columns(
            (pl.col("delivery_cost") * 1.2).alias("projected_cost")
        )
        .collect(engine="streaming")
    )

    return result


def polars_streaming_sink_large_output():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    if STREAMING_OUTPUT_PATH.exists():
        STREAMING_OUTPUT_PATH.unlink()

    (
        events
        .filter(pl.col("status") == "in_transit")
        .join(dim, on="warehouse_id", how="inner")
        .select([
            "event_id",
            "tracking_number",
            "event_ts",
            "event_date",
            "country",
            "device",
            "delivery_cost",
            "package_weight_kg",
            "warehouse_id",
            "warehouse_region",
            "warehouse_capacity",
        ])
        .with_columns(
            (pl.col("delivery_cost") * 1.2).alias("projected_cost")
        )
        .sink_parquet(STREAMING_OUTPUT_PATH)
    )

    return {
        "output_path": str(STREAMING_OUTPUT_PATH),
        "output_size_mb": round(parquet_size_mb(STREAMING_OUTPUT_PATH), 2),
    }


# Quick correctness / output-size check before benchmarking.
large_output_preview = polars_lazy_large_output()
print("Large output rows:", large_output_preview.height)
print("Large output columns:", large_output_preview.width)
display(large_output_preview.head())


Large output rows: 4001431
Large output columns: 12


event_id,tracking_number,event_ts,event_date,country,device,delivery_cost,package_weight_kg,warehouse_id,warehouse_region,warehouse_capacity,projected_cost
i64,i64,datetime[μs],date,str,str,i64,f64,i64,str,i64,f64
2,2880,2026-01-07 19:27:08,2026-01-07,"""IN""","""desktop""",8516,67.111,760,"""South""",16348,10219.2
6,154410,2026-01-06 05:20:33,2026-01-06,"""FR""","""mobile""",4169,211.618,3,"""South""",5246,5002.8
7,3571,2026-03-22 18:15:28,2026-03-22,"""UK""","""mobile""",1397,60.532,564,"""Central""",30129,1676.4
10,46592,2026-01-20 07:38:41,2026-01-20,"""US""","""mobile""",7958,167.618,615,"""Central""",31737,9549.6
12,138918,2026-03-25 03:32:52,2026-03-25,"""BR""","""desktop""",9076,19.859,277,"""North""",8355,10891.2


In [216]:
polars_execution_mode_results = []

for query_name, engine, mode, func in [
    ("Q_large_output", "Polars", "eager", polars_eager_large_output),
    ("Q_large_output", "Polars", "lazy_collect", polars_lazy_large_output),
    ("Q_large_output", "Polars", "streaming_collect", polars_streaming_collect_large_output),
    ("Q_large_output", "Polars", "streaming_sink", polars_streaming_sink_large_output),
]:
    run_benchmark(
        query_name=query_name,
        engine=engine,
        mode=mode,
        func=func,
        repetitions=3,
        data_format="parquet",
        layout="default",
        input_path=EVENTS_PATH,
        result_check="passed",
        notes=(
            "Task 3.1 Polars execution-mode comparison. "
            "The query filters in-transit deliveries, joins warehouse dimension, "
            "keeps many rows, and materializes or writes a large output."
        ),
    )
    polars_execution_mode_results.append(benchmark_results[-1])

polars_execution_mode_df = pd.DataFrame(polars_execution_mode_results, columns=BENCHMARK_COLUMNS)
display(polars_execution_mode_df)

POLARS_EXECUTION_RESULTS_PATH = OUTPUT_DIR / "polars_execution_mode_results.csv"
polars_execution_mode_df.to_csv(POLARS_EXECUTION_RESULTS_PATH, index=False)

if STREAMING_OUTPUT_PATH.exists():
    print("Streaming sink output size MB:", round(parquet_size_mb(STREAMING_OUTPUT_PATH), 2))

POLARS_EXECUTION_RESULTS_PATH

-> [Polars | eager] Q_large_output: 1.5961 s, peak RSS 14392.62 MB
-> [Polars | lazy_collect] Q_large_output: 0.9199 s, peak RSS 13441.55 MB
-> [Polars | streaming_collect] Q_large_output: 1.0917 s, peak RSS 11524.91 MB
-> [Polars | streaming_sink] Q_large_output: 1.1879 s, peak RSS 11497.09 MB


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Polars,eager,Q_large_output,parquet,default,10000000,1.5961,14392.62,209.23,passed,Task 3.1 Polars execution-mode comparison. The...
1,Polars,lazy_collect,Q_large_output,parquet,default,10000000,0.9199,13441.55,209.23,passed,Task 3.1 Polars execution-mode comparison. The...
2,Polars,streaming_collect,Q_large_output,parquet,default,10000000,1.0917,11524.91,209.23,passed,Task 3.1 Polars execution-mode comparison. The...
3,Polars,streaming_sink,Q_large_output,parquet,default,10000000,1.1879,11497.09,209.23,passed,Task 3.1 Polars execution-mode comparison. The...


Streaming sink output size MB: 89.68


PosixPath('../data/phase2_26L/group_11/polars_execution_mode_results.csv')

#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

POLARS_LIMITATION_SCENARIO = """
Polars may struggle compared to Spark when the input data, intermediate state,
or final output exceeds the memory capacity of a single machine. This is especially 
relevant for large-output queries, where the result constitutes a large fraction 
of the input and cannot be simply reduced to a small aggregated table.

In such cases, even with Polars' lazy evaluation or streaming capabilities, it may 
still need to materialize the final result within the local Python process. Spark 
becomes the more appropriate choice when the workload requires distributed memory, 
distributed shuffle operations, fault tolerance, or execution on data that is 
already stored in a distributed environment.
"""

POLARS_LIMITATION_EVIDENCE = """
In Task 3.1, the large-output query was executed to test the boundaries of local memory.
The measured execution times and memory usage (peak RSS) were:
- eager: 1.5961 s, 14392.62 MB peak RSS
- lazy_collect: 0.9199 s, 13441.55 MB peak RSS
- streaming_collect: 1.0917 s, 11524.91 MB peak RSS
- streaming_sink: 1.1879 s, 11497.09 MB peak RSS

The `eager` execution was the most memory-intensive and the slowest among in-memory operations, as it forces the engine to materialize every intermediate step. 

Utilizing `lazy_collect` provided the fastest execution time (0.9199 s) thanks to Polars' query optimizer (which applies predicate and projection pushdowns), but it still consumed a massive amount of RAM (over 13.4 GB) because the entire final output had to be materialized in memory at once.

Switching to the streaming engine successfully reduced the memory usage. The `streaming_sink` variant consumed the lowest peak RSS overall (11497.09 MB) because it processed the data in batches and streamed the results directly to disk, avoiding full memory materialization.

These results shows that Polars is exceptionally fast on a single machine, but processing large-output queries quickly pushes local RAM usage to extreme levels (nearly 14.5 GB in the worst case). If the dataset were to grow further, it would easily exceed the physical memory limits of a standard machine, making a distributed engine like Spark a necessary and more scalable alternative.
"""

display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)

**Polars limitation scenario**

Polars may struggle compared to Spark when the input data, intermediate state,
or final output exceeds the memory capacity of a single machine. This is especially 
relevant for large-output queries, where the result constitutes a large fraction 
of the input and cannot be simply reduced to a small aggregated table.

In such cases, even with Polars' lazy evaluation or streaming capabilities, it may 
still need to materialize the final result within the local Python process. Spark 
becomes the more appropriate choice when the workload requires distributed memory, 
distributed shuffle operations, fault tolerance, or execution on data that is 
already stored in a distributed environment.

**Evidence**

In Task 3.1, the large-output query was executed to test the boundaries of local memory.
The measured execution times and memory usage (peak RSS) were:
- eager: 1.5961 s, 14392.62 MB peak RSS
- lazy_collect: 0.9199 s, 13441.55 MB peak RSS
- streaming_collect: 1.0917 s, 11524.91 MB peak RSS
- streaming_sink: 1.1879 s, 11497.09 MB peak RSS

The `eager` execution was the most memory-intensive and the slowest among in-memory operations, as it forces the engine to materialize every intermediate step. 

Utilizing `lazy_collect` provided the fastest execution time (0.9199 s) thanks to Polars' query optimizer (which applies predicate and projection pushdowns), but it still consumed a massive amount of RAM (over 13.4 GB) because the entire final output had to be materialized in memory at once.

Switching to the streaming engine successfully reduced the memory footprint. The `streaming_sink` variant consumed the lowest peak RSS overall (11497.09 MB) because it processed the data in batches and streamed the results directly to disk, avoiding full memory materialization.

These results clearly shows the architectural boundary: while Polars is exceptionally fast on a single machine, processing large-output queries quickly pushes local RAM usage to extreme levels (nearly 14.5 GB in the worst case). If the dataset were to grow further, it would easily exceed the physical memory limits of a standard machine, making a distributed engine like Spark a necessary and more scalable alternative.

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [223]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

DECISION_BOUNDARY = """
For this workload, I would recommend staying with a single-node engine such as
DuckDB or Polars when the input data fits comfortably on one machine and the query
either returns a small aggregate result or a moderately sized output. In this notebook,
the dataset had 10,000,000 rows and the default Parquet file was about 209.23 MB, so
single-node tools were highly efficient.

I would switch from local Polars/DuckDB to Spark when at least one of the following
conditions is met:

1. The input data no longer fits comfortably in local RAM or local disk.
2. The query produces an output that is too large to materialize in one Python process.
3. Joins or group-by operations require a massive shuffle or intermediate state memory.
4. The workload requires fault tolerance, cluster scheduling, or repeated production execution.
5. The data is already stored in a distributed/cloud environment (like GCS or HDFS) and should be processed close to storage to avoid network bottlenecks.

For my local environment, which is equipped with 32 GB of RAM, I would be cautious when the
working set or expected output approaches the 15-20 GB range. At that point, Spark/Dataproc
may be slower for small queries because of scheduling overhead, but it becomes a much safer and inherently scalable choice.
"""

DECISION_EVIDENCE = """
The Task 2 benchmark showed that all local engines completed the standard queries on 10,000,000 rows. 
DuckDB was exceptionally fast, processing queries in under a second, while Polars also completed all variants successfully.

Task 2.5 proved that layout optimization matters before switching to Spark. For DuckDB, the `optimized_parquet` layout 
executed in just 0.7279 s and used the lowest memory (9952.78 MB), compared to 0.7749 s for the default Parquet. 
While the CSV baseline was slightly faster (0.6894 s), it consumed significantly more memory (over 10.2 GB). This shows 
that proper file formatting should be the first step before scaling up to distributed engines.

Task 3.1 demonstrated the critical difference between materializing a large output and streaming it to disk. 
The large-output query returned around 4.000,000 rows and 12 columns. The `streaming_sink` mode had the lowest peak RSS 
at 11497.09 MB, while the `eager` collection mode spiked to an extreme 14392.62 MB. 

This massive memory consumption (nearly 14.4 GB for a single operation) perfectly supports the decision boundary: 
local tools like Polars and DuckDB are blindingly fast for standard workloads, but when intermediate states or outputs 
threaten to exhaust local memory, switching to a distributed engine like Spark is necessary to prevent Out-Of-Memory crashes.
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

**Decision boundary**

For this workload, I would recommend staying with a single-node engine such as
DuckDB or Polars when the input data fits comfortably on one machine and the query
either returns a small aggregate result or a moderately sized output. In this notebook,
the dataset had 10,000,000 rows and the default Parquet file was about 209.23 MB, so
single-node tools were highly efficient.

I would switch from local Polars/DuckDB to Spark when at least one of the following
conditions is met:

1. The input data no longer fits comfortably in local RAM or local disk.
2. The query produces an output that is too large to materialize in one Python process.
3. Joins or group-by operations require a massive shuffle or intermediate state memory.
4. The workload requires fault tolerance, cluster scheduling, or repeated production execution.
5. The data is already stored in a distributed/cloud environment (like GCS or HDFS) and should be processed close to storage to avoid network bottlenecks.

For my local environment, which is equipped with 32 GB of RAM, I would be cautious when the
working set or expected output approaches the 15-20 GB range. At that point, Spark/Dataproc
may be slower for small queries because of scheduling overhead, but it becomes a much safer and inherently scalable choice.

**Evidence**

The Task 2 benchmark showed that all local engines completed the standard queries on 10,000,000 rows. 
DuckDB was exceptionally fast, processing queries in under a second, while Polars also completed all variants successfully.

Task 2.5 proved that layout optimization matters before switching to Spark. For DuckDB, the `optimized_parquet` layout 
executed in just 0.7279 s and used the lowest memory (9952.78 MB), compared to 0.7749 s for the default Parquet. 
While the CSV baseline was slightly faster (0.6894 s), it consumed significantly more memory (over 10.2 GB). This shows 
that proper file formatting should be the first step before scaling up to distributed engines.

Task 3.1 demonstrated the critical difference between materializing a large output and streaming it to disk. 
The large-output query returned around 4.000,000 rows and 12 columns. The `streaming_sink` mode had the lowest peak RSS 
at 11497.09 MB, while the `eager` collection mode spiked to an extreme 14392.62 MB. 

This massive memory consumption (nearly 14.4 GB for a single operation) perfectly supports the decision boundary: 
local tools like Polars and DuckDB are blindingly fast for standard workloads, but when intermediate states or outputs 
threaten to exhaust local memory, switching to a distributed engine like Spark is necessary to prevent Out-Of-Memory crashes.

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [224]:
# TODO: Run selected scalability experiments and append results to benchmark_results.

In [225]:
# Task 4: Thread and core scalability experiments.
# We compare DuckDB with different thread counts and PySpark with different local master settings.
# The selected query is Q3_daily_delivered_weight.

scalability_results = []


def duckdb_q3_with_threads(thread_count):
    def query():
        con = duckdb.connect(database=":memory:")
        con.execute(f"PRAGMA threads={thread_count}")

        result = con.execute(
            """
            SELECT
                event_date,
                COUNT(*) AS delivered_count,
                SUM(package_weight_kg) AS total_package_weight_kg,
                AVG(package_weight_kg) AS avg_package_weight_kg
            FROM read_parquet(?)
            WHERE status = 'delivered'
              AND event_date BETWEEN ? AND ?
            GROUP BY event_date
            ORDER BY event_date
            """,
            [str(EVENTS_PATH), DATE_FROM, DATE_TO],
        ).fetchdf()

        con.close()
        return result

    return query


for thread_count in [1, 2, 4, 8]:
    run_benchmark(
        query_name="Q3_daily_delivered_weight",
        engine="DuckDB",
        mode=f"threads_{thread_count}",
        func=duckdb_q3_with_threads(thread_count),
        repetitions=3,
        data_format="parquet",
        layout="default",
        input_path=EVENTS_PATH,
        result_check="passed",
        notes=f"Task 4 DuckDB scalability test with PRAGMA threads={thread_count}.",
    )
    scalability_results.append(benchmark_results[-1])


scalability_duckdb_df = pd.DataFrame(scalability_results, columns=BENCHMARK_COLUMNS)
display(scalability_duckdb_df)

-> [DuckDB | threads_1] Q3_daily_delivered_weight: 1.0243 s, peak RSS 9957.78 MB
-> [DuckDB | threads_2] Q3_daily_delivered_weight: 0.881 s, peak RSS 9959.55 MB
-> [DuckDB | threads_4] Q3_daily_delivered_weight: 0.8035 s, peak RSS 9964.61 MB
-> [DuckDB | threads_8] Q3_daily_delivered_weight: 0.7821 s, peak RSS 9975.13 MB


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,DuckDB,threads_1,Q3_daily_delivered_weight,parquet,default,10000000,1.0243,9957.78,209.23,passed,Task 4 DuckDB scalability test with PRAGMA thr...
1,DuckDB,threads_2,Q3_daily_delivered_weight,parquet,default,10000000,0.8810,9959.55,209.23,passed,Task 4 DuckDB scalability test with PRAGMA thr...
2,DuckDB,threads_4,Q3_daily_delivered_weight,parquet,default,10000000,0.8035,9964.61,209.23,passed,Task 4 DuckDB scalability test with PRAGMA thr...
3,DuckDB,threads_8,Q3_daily_delivered_weight,parquet,default,10000000,0.7821,9975.13,209.23,passed,Task 4 DuckDB scalability test with PRAGMA thr...


In [226]:
# Polars scalability reference.
# Polars uses a global thread pool configured at process startup.
# In this notebook we cannot safely change it without restarting the kernel,
# so we record the default process-level thread pool configuration.

polars_thread_pool_size = pl.thread_pool_size()

run_benchmark(
    query_name="Q3_daily_delivered_weight",
    engine="Polars",
    mode=f"default_thread_pool_{polars_thread_pool_size}",
    func=polars_lazy_q3_daily_delivered_weight,
    repetitions=3,
    data_format="parquet",
    layout="default",
    input_path=EVENTS_PATH,
    result_check="passed",
    notes=(
        "Task 4 Polars reference run. Polars thread pool size is configured "
        "at process startup, so it is not changed dynamically inside this notebook."
    ),
)

scalability_results.append(benchmark_results[-1])

scalability_df = pd.DataFrame(scalability_results, columns=BENCHMARK_COLUMNS)
display(scalability_df)

SCALABILITY_RESULTS_PATH = OUTPUT_DIR / "scalability_results.csv"
scalability_df.to_csv(SCALABILITY_RESULTS_PATH, index=False)
SCALABILITY_RESULTS_PATH

-> [Polars | default_thread_pool_16] Q3_daily_delivered_weight: 0.7683 s, peak RSS 10054.61 MB


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,DuckDB,threads_1,Q3_daily_delivered_weight,parquet,default,10000000,1.0243,9957.78,209.23,passed,Task 4 DuckDB scalability test with PRAGMA thr...
1,DuckDB,threads_2,Q3_daily_delivered_weight,parquet,default,10000000,0.8810,9959.55,209.23,passed,Task 4 DuckDB scalability test with PRAGMA thr...
2,DuckDB,threads_4,Q3_daily_delivered_weight,parquet,default,10000000,0.8035,9964.61,209.23,passed,Task 4 DuckDB scalability test with PRAGMA thr...
3,DuckDB,threads_8,Q3_daily_delivered_weight,parquet,default,10000000,0.7821,9975.13,209.23,passed,Task 4 DuckDB scalability test with PRAGMA thr...
4,Polars,default_thread_pool_16,Q3_daily_delivered_weight,parquet,default,10000000,0.7683,10054.61,209.23,passed,Task 4 Polars reference run. Polars thread poo...


PosixPath('../data/phase2_26L/group_11/scalability_results.csv')

### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [244]:
# TODO: Add Dataproc-specific commands, notebook cells, or instructions used by your group.
# Do not hard-code credentials or project secrets in the notebook.

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmarkLarge")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

LOCAL_BENCHMARKS_2 = [
    ("Q1_delayed_by_region", "PySpark", "local[*]", spark_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "PySpark", "local[*]", spark_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "PySpark", "local[*]", spark_q3_daily_delivered_weight),
]

for query_name, engine, mode, func in LOCAL_BENCHMARKS_2:
    run_benchmark(
        query_name=query_name,
        engine=engine,
        mode=mode,
        func=func,
        repetitions=3,
        data_format="parquet",
        layout="default",
        input_path=EVENTS_PATH,
        result_check="passed",
        notes="Dataproc benchmark on the same generated Parquet dataset. Peak memory is process RSS measured from the dataproc.",
    )

benchmark_df = pd.DataFrame(benchmark_results, columns=BENCHMARK_COLUMNS)
# print(benchmark_df)

-> [PySpark | local[*]] Q1_delayed_by_region: 0.6688 s, peak RSS 9953.48 MB
-> [PySpark | local[*]] Q2_top_expensive_in_transit: 0.6981 s, peak RSS 9953.47 MB
-> [PySpark | local[*]] Q3_daily_delivered_weight: 1.0386 s, peak RSS 9953.47 MB


In [238]:
!gcloud storage cp -q ../data/phase2_26L/group_11/events.parquet gs://tbd-2026l-11-data/ 
!gcloud storage cp -q ../data/phase2_26L/group_11/events_large.parquet gs://tbd-2026l-11-data/ 
!gcloud storage cp -q ../data/phase2_26L/group_11/dimension.parquet gs://tbd-2026l-11-data/ 
!gcloud storage cp -q ../data/phase2_26L/group_11/dimension_large.parquet gs://tbd-2026l-11-data/ 


uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://../data/phase2_26L/group_11/events.parquet to gs://tbd-2026l-11-data/events.parquet
  Completed files 5/1 | 209.2MiB/209.2MiB | 3.2MiB/s                           

Average throughput: 4.7MiB/s
uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composi

In [239]:
!gcloud storage cp dataproc-benchmark.py gs://tbd-2026l-11-code/
!gcloud dataproc jobs submit pyspark gs://tbd-2026l-11-code/dataproc-benchmark.py --cluster=tbd-cluster --region=europe-west1 --project=tbd-2026l-11 --verbosity=error

Copying file://dataproc-benchmark.py to gs://tbd-2026l-11-code/dataproc-benchmark.py
  Completed files 1/1 | 7.2kiB/7.2kiB                                          
Job [1ce389fb77384511b0fe730a021601b3] submitted.
Waiting for job output...
26/06/14 17:04:38 INFO SparkEnv: Registering MapOutputTracker
26/06/14 17:04:38 INFO SparkEnv: Registering BlockManagerMaster
26/06/14 17:04:38 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/14 17:04:38 INFO SparkEnv: Registering OutputCommitCoordinator
26/06/14 17:04:40 INFO MetricsConfig: Loaded properties from hadoop-metrics2.properties
26/06/14 17:04:40 INFO MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
26/06/14 17:04:40 INFO MetricsSystemImpl: google-hadoop-file-system metrics system started
26/06/14 17:04:40 INFO DataprocSparkPlugin: Registered 188 driver metrics
26/06/14 17:04:41 INFO DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at tbd-cluster-m.c.tbd-2026l-11.internal./10.10.10.10:803

In [240]:

!gcloud dataproc jobs submit pyspark gs://tbd-2026l-11-code/dataproc-benchmark.py --cluster=tbd-cluster --region=europe-west1 --project=tbd-2026l-11 --verbosity=error -- --stress-mode

Job [f99db593127a4775b22917441f6f39ee] submitted.
Waiting for job output...
26/06/14 17:07:09 INFO SparkEnv: Registering MapOutputTracker
26/06/14 17:07:09 INFO SparkEnv: Registering BlockManagerMaster
26/06/14 17:07:09 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/14 17:07:10 INFO SparkEnv: Registering OutputCommitCoordinator
26/06/14 17:07:11 INFO MetricsConfig: Loaded properties from hadoop-metrics2.properties
26/06/14 17:07:11 INFO MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
26/06/14 17:07:11 INFO MetricsSystemImpl: google-hadoop-file-system metrics system started
26/06/14 17:07:12 INFO DataprocSparkPlugin: Registered 188 driver metrics
26/06/14 17:07:13 INFO DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at tbd-cluster-m.c.tbd-2026l-11.internal./10.10.10.10:8032
26/06/14 17:07:13 INFO AHSProxy: Connecting to Application History server at tbd-cluster-m.c.tbd-2026l-11.internal./10.10.10.10:10200
26/06/14 17:07:14 INFO Config

#### Results and conclusion
The execution time on the Dataproc cluster is generally higher than local execution. For instance, processing the medium dataset (10,000,000 rows) took only 0.5-1s locally, compared to 6-10s on the cluster due to initialization and network overhead.

However, the cluster demonstrates a highly efficient scaling trend. When the dataset was increased by a factor of 5 (up to 50,000,000 rows), the Dataproc execution time only increased to 10–26s. Because the processing time did not increase by 5 times linearly, it shows that Spark efficiently parallelizes the workload and amortizes the initial overhead over larger datasets.

The time is higher for the Dataproc cluster because it needs to communicate with Apache Hadoop YARN (the resource manager). YARN has to find available resources across the cluster and allocate them to the task (scheduling overhead). This process takes time. Additionally, since the resources are split across different physical machines, network overhead kicks in (data shuffling).

Overall, computations for smaller datasets are easier and faster to perform on a local machine. The true advantage of cloud computing shines with massive datasets that exceed local system memory and computation power. In such cases, scalability is handled under the hood by the cloud environment. If we tried to process data of that magnitude locally, we would either run out of memory or have to process it sequentially, which would be extremely inefficient.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
TODO: Write your answer here.

Zapytania Q2_top_expensive_in_transit oraz Q1_delayed_by_region najlepiej obrazują tę różnicę. Silniki oparte na 
zapytaniach SQL lub leniwym wykonywaniu (np. DuckDB, Polars w trybie lazy) wykorzystują optymalizator do zepchnięcia filtrów (predicate pushdown) do pliku Parquet lub 
zignorowania niepotrzebnych kolumn (column pruning) przed załadowaniem danych. W efekcie DuckDB potrafi wykonać zapytanie Q1 w 0.7075s, podczas gdy domyślny Pandas 
(ładujący cały zbiór danych do pamięci przed nałożeniem maski) zajmuje 3.6353s.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

**Final answer 1**

TODO: Write your answer here.

Zapytania Q2_top_expensive_in_transit oraz Q1_delayed_by_region najlepiej obrazują tę różnicę. Silniki oparte na 
zapytaniach SQL lub leniwym wykonywaniu (np. DuckDB, Polars w trybie lazy) wykorzystują optymalizator do zepchnięcia filtrów (predicate pushdown) do pliku Parquet lub 
zignorowania niepotrzebnych kolumn (column pruning) przed załadowaniem danych. W efekcie DuckDB potrafi wykonać zapytanie Q1 w 0.2675s, podczas gdy domyślny Pandas 
(ładujący cały zbiór danych do pamięci przed nałożeniem maski) zajmuje 0.8087s.

In [2]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.

Zdecydowanie najbardziej wrażliwym na pamięć zapytaniem było Q_large_output (z Zadania 3.1). Wynika to z faktu, że zapytanie to odrzuca stosunkowo niewiele wierszy, 
tworzy złączenie, uwzględnia dwanaście kolumn i buduje gigantyczny DataFrame (ponad 4 miliony wierszy). Podniesienie i przetworzenie takiego wyniku 
skutkowało użyciem w trybie Polars eager aż 13418.41 MB szczytowej pamięci RSS, podczas gdy standardowe analityczne zapytania z głównego benchmarku zużywały ok. 9000-1000 MB.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

**Final answer 2**

TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.

Zdecydowanie najbardziej wrażliwym na pamięć zapytaniem było Q_large_output (z Zadania 3.1). Wynika to z faktu, że zapytanie to odrzuca stosunkowo niewiele wierszy, 
tworzy złączenie, uwzględnia dwanaście kolumn i buduje gigantyczny DataFrame (ponad 4 miliony wierszy). Podniesienie i przetworzenie takiego wyniku 
skutkowało użyciem w trybie Polars eager aż 13418.41 MB szczytowej pamięci RSS, podczas gdy standardowe analityczne zapytania z głównego benchmarku zużywały ok. 9000-1000 MB.

In [3]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.

Tak, leniwe wykonywanie zaoszczędziło zasoby dzięki wypchnięciu reguł filtrowania i selekcji bezpośrednio do fazy skanowania Parquet. Przykładowo, w zapytaniu Q1 
tryb Polars eager potrzebował 11246 MB w szczytowym momencie i zajmował 0.824s. Przejście na tryb Polars lazy_collect zredukowało wczytywane wiersze przed złączeniem, 
przez co zużycie pamięci spadło do 9990 MB, a czas obniżył się do 0.7137s.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

**Final answer 3**

TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.

Tak, leniwe wykonywanie zaoszczędziło zasoby dzięki wypchnięciu reguł filtrowania i selekcji bezpośrednio do fazy skanowania Parquet. Przykładowo, w zapytaniu Q1 
tryb Polars eager potrzebował 11246 MB w szczytowym momencie i zajmował 0.824s. Przejście na tryb Polars lazy_collect zredukowało wczytywane wiersze przed złączeniem, 
przez co zużycie pamięci spadło do 9990 MB, a czas obniżył się do 0.7137s.

In [4]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).

collect(engine="streaming") zredukowało w tym zapytaniu użycie pamięci (peak RSS) z 9990 MB do 9395 MB, podczas gdy czas wykonania 
lekko się pogroszył (0.8945 s vs 0.7137  w zwykłym lazy_collect).

Zasadnicza różnica między tymi dwoma mechanizmami polega na obsłudze wyniku końcowego: opcja collect(engine="streaming") pozwala na strumieniowe, 
niskopamięciowe przetwarzanie pośrednie (w paczkach), ale na samym końcu i tak musi w całości zmaterializować wynikowy DataFrame w pamięci RAM procesu Pythona. 
Z kolei sink_parquet() przetwarza paczki danych i natychmiast zrzuca je na dysk (z pominięciem trzymania ich w RAM-ie), co jest kluczowe i bezpieczniejsze, 
gdy rozmiar samego wyniku przekracza możliwości lokalnej pamięci operacyjnej.
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

**Final answer 4**

TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).

collect(engine="streaming") zredukowało w tym zapytaniu użycie pamięci (peak RSS) z 9990 MB do 9395 MB, podczas gdy czas wykonania 
lekko się pogroszył (0.8945 s vs 0.7137  w zwykłym lazy_collect).

Zasadnicza różnica między tymi dwoma mechanizmami polega na obsłudze wyniku końcowego: opcja collect(engine="streaming") pozwala na strumieniowe, 
niskopamięciowe przetwarzanie pośrednie (w paczkach), ale na samym końcu i tak musi w całości zmaterializować wynikowy DataFrame w pamięci RAM procesu Pythona. 
Z kolei sink_parquet() przetwarza paczki danych i natychmiast zrzuca je na dysk (z pominięciem trzymania ich w RAM-ie), co jest kluczowe i bezpieczniejsze, 
gdy rozmiar samego wyniku przekracza możliwości lokalnej pamięci operacyjnej.

In [5]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.

Strumieniowanie do pliku (streaming sink) było o wiele bardziej odpowiednie niż zbieranie wyniku (collect), ponieważ wyjściowy zbiór danych
był bardzo duży - liczył 12 kolumn, ponad 4 miliony wierszy i ważył 89.66 MB na dysku. Każda opcja używająca collect wymuszała ostateczną materializację 
tego ogromnego wyniku w pamięci głównej procesu Pythona, co pochłaniało ponad 10 000 MB RAM-u. Zastosowanie streaming sink omija całkowicie materializację 
pełnego wyniku w przestrzeni Pythona, zapisując dane w paczkach bezpośrednio na dysk, co było najbezpieczniejszą opcją z perspektywy limitów pamięciowych.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

**Final answer 5**

TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.

Strumieniowanie do pliku (streaming sink) było o wiele bardziej odpowiednie niż zbieranie wyniku (collect), ponieważ wyjściowy zbiór danych
był bardzo duży - liczył 12 kolumn, ponad 4 miliony wierszy i ważył 89.66 MB na dysku. Każda opcja używająca collect wymuszała ostateczną materializację 
tego ogromnego wyniku w pamięci głównej procesu Pythona, co pochłaniało ponad 10 000 MB RAM-u. Zastosowanie streaming sink omija całkowicie materializację 
pełnego wyniku w przestrzeni Pythona, zapisując dane w paczkach bezpośrednio na dysk, co było najbezpieczniejszą opcją z perspektywy limitów pamięciowych.

In [6]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.

Tak, wyniki lokalnego Sparka pokrywają się w większości z standardowymi jedno wątkowymi silnikami na tych zestawach danych (10 milionów wierszy, ok. 209 MB w formacie Parquet). PySpark zajmował w benchmarku od ok. 0.63 s do 0.64 s dla zapytań Q1-Q2. Jest to generalnie 
wynik porównywalny, a nawet lepszy. Jednak różnice widać w zapytaniu Q3 - 0.98s do pysparka vs. 0.7 dla DuckDB, taki wynik może być spowodowany narzutem architektury maszyny wirtualnej Javy (JVM), 
tworzenia planu, logiki harmonogramowania (scheduling overhead) oraz mechanizmów shuffle, które zaprojektowane są pod dane rozproszone na klastrze.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

**Final answer 6**

TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.

Tak, wyniki lokalnego Sparka pokrywają się w większości z standardowymi jedno wątkowymi silnikami na tych zestawach danych (10 milionów wierszy, ok. 209 MB w formacie Parquet). PySpark zajmował w benchmarku od ok. 0.63 s do 0.64 s dla zapytań Q1-Q2. Jest to generalnie 
wynik porównywalny, a nawet lepszy. Jednak różnice widać w zapytaniu Q3 - 0.98s do pysparka vs. 0.7 dla DuckDB, taki wynik może być spowodowany narzutem architektury maszyny wirtualnej Javy (JVM), 
tworzenia planu, logiki harmonogramowania (scheduling overhead) oraz mechanizmów shuffle, które zaprojektowane są pod dane rozproszone na klastrze.

In [7]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
Decyzję o przeniesieniu przetwarzania z maszyny lokalnej na klaster w chmurze podjąłbym na podstawie dwóch głównych czynników: rozmiaru danych oraz kształtu zapytania (query shape).

1. Rozmiar zbioru danych:
Rozważyłbym migrację do chmury, gdy zbiór danych przekracza 50 000 000 rekordów lub gdy jego rozmiar po dekompresji zbliża się do limitu mojej pamięci RAM (32 GB). Po przekroczeniu tej granicy lokalny silnik często używa dysku jako dodatkowej pamięci systemowej (użycie partycji swap). W takim scenariuszu narzut czasowy klastra (YARN Scheduling Overhead) staje się marginalny w porównaniu do spadku wydajności lokalnej maszyny z powolnymi operacjami wejścia/wyjścia (I/O). 

2. Kształt zapytania:
Nawet przy mniejszych zbiorach danych przeszedłbym na klaster, jeśli zapytanie wymagałoby potężnego tasowania danych (Shuffle). Operacje takie jak wielokrotne łączenie dużych tabel (JOIN), skomplikowane grupowania (GROUP BY) czy agregacje oparte na oknach czasowych zużywają większą ilość pamięci systemowej. Klaster Spark skaluje takie operacje horyzontalnie, rozdzielając pracę na wiele węzłów roboczych.

3. Zużycie sprzętu:
Używanie lokalnego laptopa do zadań przekraczających pamięć RAM generuje gigantyczną ilość zapisów na dysku. Dyski SSD nie są projektowane pod taką ilość odczytów i zapisów, co prowadzi do ich szybkiej degradacji. Dodatkowo, przetwarzanie na klastrze zapewnia nam niezawodnosć - jeśli jeden węzeł (Worker) ulegnie awarii z powodu braku pamięci, zadanie zostanie po prostu przerzucone na inną maszynę. Na komputerze lokalnym taki sam problem kończy się błędem i utratą godzin pracy.
"""
display_answer("Final answer 7", FINAL_ANSWER_7)

**Final answer 7**

Decyzję o przeniesieniu przetwarzania z maszyny lokalnej na klaster w chmurze podjąłbym na podstawie dwóch głównych czynników: rozmiaru danych oraz kształtu zapytania (query shape).

1. Rozmiar zbioru danych:
Rozważyłbym migrację do chmury, gdy zbiór danych przekracza 50 000 000 rekordów lub gdy jego rozmiar po dekompresji zbliża się do limitu mojej pamięci RAM (32 GB). Po przekroczeniu tej granicy lokalny silnik często używa dysku jako dodatkowej pamięci systemowej (użycie partycji swap). W takim scenariuszu narzut czasowy klastra (YARN Scheduling Overhead) staje się marginalny w porównaniu do spadku wydajności lokalnej maszyny z powolnymi operacjami wejścia/wyjścia (I/O). 

2. Kształt zapytania:
Nawet przy mniejszych zbiorach danych przeszedłbym na klaster, jeśli zapytanie wymagałoby potężnego tasowania danych (Shuffle). Operacje takie jak wielokrotne łączenie dużych tabel (JOIN), skomplikowane grupowania (GROUP BY) czy agregacje oparte na oknach czasowych zużywają większą ilość pamięci systemowej. Klaster Spark skaluje takie operacje horyzontalnie, rozdzielając pracę na wiele węzłów roboczych.

3. Zużycie sprzętu:
Używanie lokalnego laptopa do zadań przekraczających pamięć RAM generuje gigantyczną ilość zapisów na dysku. Dyski SSD nie są projektowane pod taką ilość odczytów i zapisów, co prowadzi do ich szybkiej degradacji. Dodatkowo, przetwarzanie na klastrze zapewnia nam niezawodnosć - jeśli jeden węzeł (Worker) ulegnie awarii z powodu braku pamięci, zadanie zostanie po prostu przerzucone na inną maszynę. Na komputerze lokalnym taki sam problem kończy się błędem i utratą godzin pracy.

In [8]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
TODO: Write your answer here. Mention runtime, memory, dtypes, and whether string-heavy or IO-heavy queries changed the result.

Backend używający typu pyarrow udowodnił poprawę efektywności zarówno w czasie ładowania (IO) plików Parquet, jak i w ogólnej optymalizacji typów tekstowych.

Znakomicie widać to na dwóch przykładach z benchmarku:

Dla zapytań obciążających IO (IO-heavy): Czas trwania zapytania Q3 z użyciem klasycznego silnika NumPy wziętego jako pd.read_parquet(path) 
zajmował aż 4.26s i zużywał w szczycie 11487 MB RAM. Ta sama akcja wykonana z przełącznikami engine="pyarrow", dtype_backend="pyarrow" 
zajęła tylko 1.54s pochłaniając 9567 MB.

Dla zapytań tekstowych (String-heavy): W zapytaniu Q1, które intensywnie operuje na tekstach (filtrowanie stringów, złączenie i grupowanie po regionie), 
czas wykonania spadł z 3.63s (i 10915 MB) w domyślnym Pandasie do zaledwie 0.82s (i 10285 MB RAM) po użyciu backendu PyArrow.

Tak wyraźna optymalizacja wynika z faktu, że PyArrow redukuje kosztowne narzuty konwersji pomiędzy kolumnowym formatem Parquet a klasycznym układem Python/NumPy, 
zachowując o wiele sprawniejsze alokowanie stringów w pamięci.
"""
display_answer("Final answer 8", FINAL_ANSWER_8)


**Final answer 8**

TODO: Write your answer here. Mention runtime, memory, dtypes, and whether string-heavy or IO-heavy queries changed the result.

Backend używający typu pyarrow udowodnił poprawę efektywności zarówno w czasie ładowania (IO) plików Parquet, jak i w ogólnej optymalizacji typów tekstowych.

Znakomicie widać to na dwóch przykładach z benchmarku:

Dla zapytań obciążających IO (IO-heavy): Czas trwania zapytania Q3 z użyciem klasycznego silnika NumPy wziętego jako pd.read_parquet(path) 
zajmował aż 4.26s i zużywał w szczycie 11487 MB RAM. Ta sama akcja wykonana z przełącznikami engine="pyarrow", dtype_backend="pyarrow" 
zajęła tylko 1.54s pochłaniając 9567 MB.

Dla zapytań tekstowych (String-heavy): W zapytaniu Q1, które intensywnie operuje na tekstach (filtrowanie stringów, złączenie i grupowanie po regionie), 
czas wykonania spadł z 3.63s (i 10915 MB) w domyślnym Pandasie do zaledwie 0.82s (i 10285 MB RAM) po użyciu backendu PyArrow.

Tak wyraźna optymalizacja wynika z faktu, że PyArrow redukuje kosztowne narzuty konwersji pomiędzy kolumnowym formatem Parquet a klasycznym układem Python/NumPy, 
zachowując o wiele sprawniejsze alokowanie stringów w pamięci.